# 03 — Modeling

目標：訓練多個模型、交叉驗證比較 PR-AUC，儲存最佳模型。

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import average_precision_score
import warnings
warnings.filterwarnings('ignore')

## 1. 載入處理後的資料

In [ ]:
train = pd.read_csv('../data/processed/train.csv')
test  = pd.read_csv('../data/processed/test.csv')

X_train = train.drop(columns=['Class'])
y_train = train['Class']
X_test  = test.drop(columns=['Class'])
y_test  = test['Class']

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. 定義模型

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='aucpr', verbosity=0),
}

## 3. 交叉驗證 (PR-AUC)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=-1)
    results[name] = scores
    print(f'{name}: PR-AUC = {scores.mean():.4f} ± {scores.std():.4f}')

## 4. 選出最佳模型並訓練全量資料

In [ ]:
best_name = max(results, key=lambda k: results[k].mean())
print(f'Best model: {best_name}')

best_model = models[best_name]
best_model.fit(X_train, y_train)

# 測試集 PR-AUC
y_prob = best_model.predict_proba(X_test)[:, 1]
test_prauc = average_precision_score(y_test, y_prob)
print(f'Test PR-AUC: {test_prauc:.4f}')

## 5. 儲存模型

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
print(f'Saved: models/best_model.pkl  ({best_name})')